# 基于预训练模型的细胞类型注释微调

本教程演示如何在新数据集上微调预训练模型以完成细胞类型注释任务。我们以多发性硬化症数据集为例，在预训练的全身模型上进行微调。请从 https://drive.google.com/drive/folders/1Qd42YNabzyr2pWt9xoY4cVMTAxsNBt4v?usp=sharing 下载数据集文件夹。

我们将微调流程总结为以下步骤，可作为细胞类型注释任务及其他任务微调的通用方法：

     1. 指定集成任务的超参数设置
     
     2. 加载和预处理数据
     
     3. 加载预训练的scGPT模型
     
     4. 使用特定任务目标微调scGPT
     
     5. 评估微调后的scGPT

In [1]:
# 导入必要的库
import copy
import gc
import json
import os
from pathlib import Path
import shutil
import sys
import time
import traceback
from typing import List, Tuple, Dict, Union, Optional
import warnings
import pandas as pd
import pickle
import torch
from anndata import AnnData
import scanpy as sc
import scvi
import seaborn as sns
import numpy as np
import wandb
from scipy.sparse import issparse
import matplotlib.pyplot as plt
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.metrics import confusion_matrix

# 添加scGPT路径
sys.path.insert(0, "../")
import scgpt as scg
from scgpt.model import TransformerModel, AdversarialDiscriminator
from scgpt.tokenizer import tokenize_and_pad_batch, random_mask_value
from scgpt.loss import (
    masked_mse_loss,
    masked_relative_error,
    criterion_neg_log_bernoulli,
)
from scgpt.tokenizer.gene_tokenizer import GeneVocab
from scgpt.preprocess import Preprocessor
from scgpt import SubsetsBatchSampler
from scgpt.utils import set_seed, category_str2int, eval_scib_metrics

# 设置绘图参数
sc.set_figure_params(figsize=(6, 6))
os.environ["KMP_WARNINGS"] = "off"
warnings.filterwarnings('ignore')

print("✅ 所有库加载完成！")

## 步骤1：指定细胞类型注释任务的超参数设置

In [2]:
# 设置随机种子确保可重复性
set_seed(42)

# 超参数设置
params = {
    "model_dir": Path("../save/scGPT_human"),  # 预训练模型路径
    "data_dir": Path("../data/MS"),  # 数据集路径
    "save_dir": Path("../results/MS_annotation"),  # 结果保存路径
    "batch_size": 32,  # 批次大小
    "lr": 1e-4,  # 学习率
    "epochs": 10,  # 训练轮数
    "num_workers": 4,  # 数据加载器的工作进程数
    "max_grad_norm": 1.0,  # 梯度裁剪的最大范数
    "patience": 10,  # 早停耐心值
}

# 创建保存目录
params["save_dir"].mkdir(parents=True, exist_ok=True)

print("✅ 超参数设置完成！")

## 步骤2：加载和预处理数据

In [3]:
# 加载数据集
adata = sc.read_h5ad(params["data_dir"] / "MS.h5ad")

# 查看数据基本信息
print(f"数据形状: {adata.shape}")
print(f"细胞数量: {adata.n_obs}")
print(f"基因数量: {adata.n_vars}")
print(f"细胞类型: {adata.obs['cell_type'].unique()}")

# 数据预处理
# 1. 过滤低质量细胞
sc.pp.filter_cells(adata, min_genes=200)
# 2. 过滤低表达基因
sc.pp.filter_genes(adata, min_cells=3)
# 3. 归一化
sc.pp.normalize_total(adata, target_sum=1e4)
# 4. log转换
sc.pp.log1p(adata)

print("✅ 数据预处理完成！")

## 步骤3：加载预训练的scGPT模型

In [4]:
# 加载预训练模型配置
model_config_file = params["model_dir"] / "args.json"
with open(model_config_file, "r") as f:
    model_args = json.load(f)

# 加载词汇表
vocab_file = params["model_dir"] / "vocab.json"
with open(vocab_file, "r") as f:
    vocab_dict = json.load(f)
vocab = GeneVocab.from_dict(vocab_dict)

# 创建模型
model = TransformerModel(
    ntoken=len(vocab),
    d_model=model_args.get("embsize", 128),
    nhead=model_args.get("num_heads", 4),
    d_hid=model_args.get("d_ffn", 128),
    nlayers=model_args.get("num_layers", 4),
    vocab=vocab,
    use_fast_transformer=False,  # Intel XPU不支持Flash Attention
)

# 加载预训练权重
model_file = params["model_dir"] / "best_model.pt"
model_state = torch.load(model_file, map_location="cpu")
model.load_state_dict(model_state, strict=False)

print("✅ 预训练模型加载完成！")

## 步骤4：使用特定任务目标微调scGPT

In [5]:
# 添加分类头用于细胞类型预测
num_cell_types = len(adata.obs["cell_type"].unique())
classifier = nn.Linear(model_args.get("embsize", 128), num_cell_types)

# 将模型和分类器移动到设备
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
classifier = classifier.to(device)

# 定义优化器
optimizer = torch.optim.Adam(
    list(model.parameters()) + list(classifier.parameters()),
    lr=params["lr"]
)

# 定义损失函数
criterion = nn.CrossEntropyLoss()

# 编码细胞类型标签
le = sklearn.preprocessing.LabelEncoder()
adata.obs["cell_type_code"] = le.fit_transform(adata.obs["cell_type"])

print("✅ 模型准备完成！")

In [6]:
# 划分训练集和测试集
train_idx, test_idx = train_test_split(
    np.arange(adata.n_obs),
    test_size=0.2,
    stratify=adata.obs["cell_type_code"]
)

# 创建数据集类
class CellDataset(Dataset):
    def __init__(self, adata, idx):
        self.adata = adata[idx]
        self.labels = torch.tensor(self.adata.obs["cell_type_code"].values, dtype=torch.long)
        
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        # 获取细胞表达数据
        expr = self.adata.X[idx].toarray() if issparse(self.adata.X) else self.adata.X[idx]
        return torch.tensor(expr, dtype=torch.float32), self.labels[idx]

# 创建数据加载器
train_dataset = CellDataset(adata, train_idx)
test_dataset = CellDataset(adata, test_idx)

train_loader = DataLoader(
    train_dataset,
    batch_size=params["batch_size"],
    shuffle=True,
    num_workers=params["num_workers"]
)

test_loader = DataLoader(
    test_dataset,
    batch_size=params["batch_size"],
    shuffle=False,
    num_workers=params["num_workers"]
)

print("✅ 数据加载器创建完成！")

In [7]:
# 训练循环
best_acc = 0.0
patience_counter = 0

for epoch in range(params["epochs"]):
    model.train()
    classifier.train()
    train_loss = 0.0
    
    for batch in train_loader:
        inputs, labels = batch
        inputs = inputs.to(device)
        labels = labels.to(device)
        
        # 前向传播
        outputs = model(inputs)
        cls_emb = outputs[:, 0, :]  # 获取[CLS]token的嵌入
        logits = classifier(cls_emb)
        
        # 计算损失
        loss = criterion(logits, labels)
        
        # 反向传播
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), params["max_grad_norm"])
        optimizer.step()
        
        train_loss += loss.item()
    
    # 验证
    model.eval()
    classifier.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for batch in test_loader:
            inputs, labels = batch
            inputs = inputs.to(device)
            labels = labels.to(device)
            
            outputs = model(inputs)
            cls_emb = outputs[:, 0, :]
            logits = classifier(cls_emb)
            
            _, predicted = torch.max(logits.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    acc = correct / total
    print(f"Epoch {epoch+1}/{params['epochs']} | 训练损失: {train_loss/len(train_loader):.4f} | 测试准确率: {acc:.4f}")
    
    # 保存最佳模型
    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), params["save_dir"] / "best_model.pt")
        torch.save(classifier.state_dict(), params["save_dir"] / "best_classifier.pt")
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= params["patience"]:
            print("早停！")
            break

print(f"✅ 训练完成！最佳准确率: {best_acc:.4f}")

## 步骤5：评估微调后的scGPT

In [8]:
# 加载最佳模型
model.load_state_dict(torch.load(params["save_dir"] / "best_model.pt"))
classifier.load_state_dict(torch.load(params["save_dir"] / "best_classifier.pt"))

# 在测试集上进行预测
model.eval()
classifier.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        inputs, labels = batch
        inputs = inputs.to(device)
        
        outputs = model(inputs)
        cls_emb = outputs[:, 0, :]
        logits = classifier(cls_emb)
        
        _, predicted = torch.max(logits.data, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

# 计算评估指标
all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

accuracy = sklearn.metrics.accuracy_score(all_labels, all_preds)
f1_macro = sklearn.metrics.f1_score(all_labels, all_preds, average="macro")

print(f"准确率: {accuracy:.4f}")
print(f"F1分数（macro）: {f1_macro:.4f}")

# 绘制混淆矩阵
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel("预测标签")
plt.ylabel("真实标签")
plt.title("细胞类型分类混淆矩阵")
plt.show()

print("✅ 评估完成！")